# **Notebook Training Model AI Deteksi Kebocoran Pipa Air Asrama**

### **Import Libraries**

In [1]:
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

### **Load Dataset**

In [2]:
telemetry = pd.read_csv("data/telemetry_data.csv", parse_dates=["recorded_at"])
telemetry.head()

,id,node_id,flow_rate_lpm,recorded_at
0,1c080f39-71a3-440c-b318-ed739d686a01,ae34f1c8-19aa-4314-9ac2-c0b7db02fc84,10,2026-05-01 10:28:00+00:00
1,32fdd1fb-12cc-4df5-ad09-ed5834d34b87,ae34f1c8-19aa-4314-9ac2-c0b7db02fc84,9,2026-05-01 10:29:00+00:00
2,06aad90b-45c3-442f-878c-f3f9c90dff74,ae34f1c8-19aa-4314-9ac2-c0b7db02fc84,9,2026-05-01 10:30:00+00:00
3,9ca1812e-deee-40b9-98e9-66ad060e77d9,ae34f1c8-19aa-4314-9ac2-c0b7db02fc84,11,2026-05-01 10:31:00+00:00
4,5a878e52-81c6-4b12-a070-b68a41ab6f17,ae34f1c8-19aa-4314-9ac2-c0b7db02fc84,11,2026-05-01 10:32:00+00:00


In [3]:
anomalies = pd.read_csv("data/anomalies.csv", parse_dates=["start_time", "end_time"])
anomalies.head()

,anomaly_id,node_id,start_time,end_time,ai_score,is_resolved
0,1085c8f4-5178-455e-a2a7-1b1b3d6df16b,502a433d-da06-4826-9b9d-a365e42a0da5,2026-05-06 08:02:00+00:00,2026-05-06 08:51:00+00:00,0.981502,False
1,2b1535ed-8500-4027-a812-98c5818c9a32,502a433d-da06-4826-9b9d-a365e42a0da5,2026-05-06 04:14:00+00:00,2026-05-06 05:50:00+00:00,0.756445,True
2,9ac122b0-4bfa-434d-b06b-06696364bac5,e32a28bd-b7c2-4f48-ad44-57750ac65b82,2026-05-01 17:49:00+00:00,2026-05-01 18:59:00+00:00,0.898084,False
3,e7edcc88-2463-43b2-9182-93c8f926e4e6,03863fdb-16e1-4842-9487-6fc48dcf7f69,2026-05-08 04:50:00+00:00,2026-05-08 05:45:00+00:00,0.807433,True


In [4]:
nodes = pd.read_csv("data/nodes.csv", parse_dates=["last_sync"])
nodes.head()

,node_id,location_block,is_online,last_sync
0,ae34f1c8-19aa-4314-9ac2-c0b7db02fc84,B2,True,2026-05-08 10:22:00+00:00
1,502a433d-da06-4826-9b9d-a365e42a0da5,C1,True,2026-05-08 10:02:00+00:00
2,e32a28bd-b7c2-4f48-ad44-57750ac65b82,A1,False,2026-05-08 10:16:00+00:00
3,d3e1d839-f50b-4c29-b221-6c4782c5087b,B2,True,2026-05-08 09:44:00+00:00
4,03863fdb-16e1-4842-9487-6fc48dcf7f69,A1,True,2026-05-08 09:57:00+00:00


### **Data Preprocessing**

`Feature Engineering`

In [5]:
# urutkan data berdasarkan node_id dan recorded_at untuk memastikan label anomali yang benar
telemetry = telemetry.sort_values(["node_id", "recorded_at"]).reset_index(drop=True)
telemetry.head()

,id,node_id,flow_rate_lpm,recorded_at
0,e2121da0-4cb7-4439-beb7-c5a835c8740d,03863fdb-16e1-4842-9487-6fc48dcf7f69,8,2026-05-01 10:28:00+00:00
1,43853708-3be8-4364-a361-8e6279850cff,03863fdb-16e1-4842-9487-6fc48dcf7f69,11,2026-05-01 10:29:00+00:00
2,d03e3a46-a3a2-4392-bc34-4b49aa24a9c5,03863fdb-16e1-4842-9487-6fc48dcf7f69,7,2026-05-01 10:30:00+00:00
3,6b3730d5-77b1-4887-b49c-1ce31b1345de,03863fdb-16e1-4842-9487-6fc48dcf7f69,9,2026-05-01 10:31:00+00:00
4,1e21a4db-d0d1-4f07-909f-602dcad3fd9e,03863fdb-16e1-4842-9487-6fc48dcf7f69,9,2026-05-01 10:32:00+00:00


In [6]:
# buat label anomali dari rentang waktu pada tabel anomalies
telemetry["label"] = 0

for _, row in anomalies.iterrows():
    mask = (
        (telemetry["node_id"] == row["node_id"])
        & (telemetry["recorded_at"] >= row["start_time"])
        & (telemetry["recorded_at"] <= row["end_time"])
    )
    telemetry.loc[mask, "label"] = 1

telemetry.head()

,id,node_id,flow_rate_lpm,recorded_at,label
0,e2121da0-4cb7-4439-beb7-c5a835c8740d,03863fdb-16e1-4842-9487-6fc48dcf7f69,8,2026-05-01 10:28:00+00:00,0
1,43853708-3be8-4364-a361-8e6279850cff,03863fdb-16e1-4842-9487-6fc48dcf7f69,11,2026-05-01 10:29:00+00:00,0
2,d03e3a46-a3a2-4392-bc34-4b49aa24a9c5,03863fdb-16e1-4842-9487-6fc48dcf7f69,7,2026-05-01 10:30:00+00:00,0
3,6b3730d5-77b1-4887-b49c-1ce31b1345de,03863fdb-16e1-4842-9487-6fc48dcf7f69,9,2026-05-01 10:31:00+00:00,0
4,1e21a4db-d0d1-4f07-909f-602dcad3fd9e,03863fdb-16e1-4842-9487-6fc48dcf7f69,9,2026-05-01 10:32:00+00:00,0


In [7]:
# feature engineering untuk menangkap pola lokal per node
# flow_rate_lpm: nilai asli sebagai sinyal utama
telemetry["flow_rate_lpm"] = telemetry["flow_rate_lpm"].astype(float)

# flow_mean_30: rata-rata bergulir 30 menit untuk baseline lokal
telemetry["flow_mean_30"] = telemetry.groupby("node_id")["flow_rate_lpm"].transform(
    lambda s: s.rolling(30, min_periods=1).mean()
 )

# flow_std_30: variasi lokal; lonjakan anomali biasanya menaikkan variasi
telemetry["flow_std_30"] = telemetry.groupby("node_id")["flow_rate_lpm"].transform(
    lambda s: s.rolling(30, min_periods=1).std().fillna(0)
 )

# flow_median_30: median lokal yang lebih tahan terhadap outlier
telemetry["flow_median_30"] = telemetry.groupby("node_id")["flow_rate_lpm"].transform(
    lambda s: s.rolling(30, min_periods=1).median()
 )

# flow_max_30 dan flow_min_30: batas atas/bawah lokal untuk menangkap lonjakan/penurunan
telemetry["flow_max_30"] = telemetry.groupby("node_id")["flow_rate_lpm"].transform(
    lambda s: s.rolling(30, min_periods=1).max()
 )
telemetry["flow_min_30"] = telemetry.groupby("node_id")["flow_rate_lpm"].transform(
    lambda s: s.rolling(30, min_periods=1).min()
 )

# flow_delta: perubahan antar menit untuk mendeteksi lonjakan tiba-tiba
telemetry["flow_delta"] = telemetry.groupby("node_id")["flow_rate_lpm"].diff().fillna(0)

# z_score_30: deviasi terhadap baseline lokal
safe_std = telemetry["flow_std_30"].replace(0, np.nan)
telemetry["z_score_30"] = ((telemetry["flow_rate_lpm"] - telemetry["flow_mean_30"]) / safe_std).fillna(0)

# daftar fitur untuk training model
feature_cols = [
    "flow_rate_lpm",
    "flow_mean_30",
    "flow_std_30",
    "flow_median_30",
    "flow_max_30",
    "flow_min_30",
    "flow_delta",
    "z_score_30",
]
telemetry[feature_cols + ["label"]].head()

,flow_rate_lpm,flow_mean_30,flow_std_30,flow_median_30,flow_max_30,flow_min_30,flow_delta,z_score_30,label
0,8.0,8.000000,0.000000,8.0,8.0,8.0,0.0,0.000000,0
1,11.0,9.500000,2.121320,9.5,11.0,8.0,3.0,0.707107,0
2,7.0,8.666667,2.081666,8.0,11.0,7.0,-4.0,-0.800641,0
3,9.0,8.750000,1.707825,8.5,11.0,7.0,2.0,0.146385,0
4,9.0,8.800000,1.483240,9.0,11.0,7.0,0.0,0.134840,0


`Train-Val-Test Split`

In [8]:
# split berbasis waktu (70% train, 15% validation, 15% test)
train_split = telemetry["recorded_at"].quantile(0.70)
val_split = telemetry["recorded_at"].quantile(0.85)

In [9]:
# split data train/val/test secara kronologis
train_df = telemetry[telemetry["recorded_at"] <= train_split]
train_df[feature_cols + ["label"]].head()

,flow_rate_lpm,flow_mean_30,flow_std_30,flow_median_30,flow_max_30,flow_min_30,flow_delta,z_score_30,label
0,8.0,8.000000,0.000000,8.0,8.0,8.0,0.0,0.000000,0
1,11.0,9.500000,2.121320,9.5,11.0,8.0,3.0,0.707107,0
2,7.0,8.666667,2.081666,8.0,11.0,7.0,-4.0,-0.800641,0
3,9.0,8.750000,1.707825,8.5,11.0,7.0,2.0,0.146385,0
4,9.0,8.800000,1.483240,9.0,11.0,7.0,0.0,0.134840,0


In [10]:
val_df = telemetry[(telemetry["recorded_at"] > train_split) & (telemetry["recorded_at"] <= val_split)]
val_df[feature_cols + ["label"]].head()

,flow_rate_lpm,flow_mean_30,flow_std_30,flow_median_30,flow_max_30,flow_min_30,flow_delta,z_score_30,label
7056,7.0,7.033333,1.188547,7.0,10.0,4.0,-1.0,-0.028045,0
7057,7.0,7.000000,1.174440,7.0,10.0,4.0,0.0,0.000000,0
7058,9.0,7.033333,1.217214,7.0,10.0,4.0,2.0,1.615712,0
7059,7.0,7.066667,1.201532,7.0,10.0,4.0,-2.0,-0.055485,0
7060,6.0,7.066667,1.201532,7.0,10.0,4.0,-1.0,-0.887756,0


In [11]:
test_df = telemetry[telemetry["recorded_at"] > val_split]
test_df[feature_cols + ["label"]].head()

,flow_rate_lpm,flow_mean_30,flow_std_30,flow_median_30,flow_max_30,flow_min_30,flow_delta,z_score_30,label
8568,6.0,7.900000,1.213431,8.0,11.0,6.0,-4.0,-1.565809,0
8569,8.0,7.933333,1.201532,8.0,11.0,6.0,2.0,0.055485,0
8570,11.0,8.033333,1.325697,8.0,11.0,6.0,3.0,2.237817,0
8571,9.0,8.066667,1.337350,8.0,11.0,6.0,-2.0,0.697897,0
8572,7.0,8.033333,1.351457,8.0,11.0,6.0,-2.0,-0.764607,0


In [12]:
# ekstrak fitur dan label per split
X_train = train_df[feature_cols].to_numpy()
y_train = train_df["label"].to_numpy()
X_val = val_df[feature_cols].to_numpy()
y_val = val_df["label"].to_numpy()
X_test = test_df[feature_cols].to_numpy()
y_test = test_df["label"].to_numpy()

In [13]:
# distribusi label di setiap split
label_dist = pd.DataFrame(
    {
        "train": train_df["label"].value_counts(),
        "val": val_df["label"].value_counts(),
        "test": test_df["label"].value_counts(),
    }
).fillna(0).astype(int).sort_index()

label_dist

,train,val,test
label,,,
0,35110,7512,7504
1,170,48,56


### **Modelling**

`Train model Isolation Forest`

In [14]:
# perkiraan proporsi anomali dari data latih untuk contamination
contamination = float(max(0.003, y_train.mean()))

# pipeline standardisasi + Isolation Forest
model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "iforest",
            IsolationForest(
                n_estimators=200,
                contamination=contamination,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

# latih hanya pada data normal jika label tersedia
fit_X = X_train[y_train == 0] if y_train.sum() > 0 else X_train
model.fit(fit_X)

Pipeline(steps=[('scaler', StandardScaler()),
                ('iforest',
                 IsolationForest(contamination=0.00481859410430839,
                                 n_estimators=200, n_jobs=-1,
                                 random_state=42))])

`Evaluation`

In [15]:
# prediksi default dari model (threshold bawaan Isolation Forest)
y_pred = (model.predict(X_test) == -1).astype(int)
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

# hitung TPR/FPR pada test
tpr = tp / (tp + fn) if (tp + fn) else 0.0
fpr = fp / (fp + tn) if (fp + tn) else 0.0

print(f"TPR (Recall anomali): {tpr:.4f}")
print(f"FPR: {fpr:.4f}")

# metrik AUC untuk melihat kualitas ranking skor anomali
anomaly_score = -model.decision_function(X_test)
roc_auc = roc_auc_score(y_test, anomaly_score)
pr_auc = average_precision_score(y_test, anomaly_score)
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC: {pr_auc:.4f}")

print(classification_report(y_test, y_pred, digits=4))

TPR (Recall anomali): 1.0000
FPR: 0.0076
ROC-AUC: 0.9975
PR-AUC: 0.5696
              precision    recall  f1-score   support

           0     1.0000    0.9924    0.9962      7504
           1     0.4956    1.0000    0.6627        56

    accuracy                         0.9925      7560
   macro avg     0.7478    0.9962    0.8295      7560
weighted avg     0.9963    0.9925    0.9937      7560



`Threshold Tuning agar memenuhi target TPR > 95% & FPR < 5%`

In [16]:
# cari threshold di validation agar memenuhi target TPR/FPR
best_threshold = None
val_score = -model.decision_function(X_val)
thresholds = np.quantile(val_score, np.linspace(0.5, 0.995, 200))
rows = []

for thr in thresholds:
    y_hat = (val_score >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, y_hat, labels=[0, 1]).ravel()
    tpr = tp / (tp + fn) if (tp + fn) else 0.0
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    rows.append({"threshold": float(thr), "tpr": tpr, "fpr": fpr})

results = pd.DataFrame(rows)
candidates = results[(results["tpr"] >= 0.95) & (results["fpr"] <= 0.05)]
if candidates.empty:
    print("Tidak ada threshold yang memenuhi TPR > 0.95 and FPR < 0.05 untuk validation")
else:
    best = candidates.sort_values(["tpr", "fpr"], ascending=[False, True]).iloc[0]
    best_threshold = float(best["threshold"])
    print(f"Best threshold (val): {best_threshold:.6f}")
    print(f"Validation TPR: {best['tpr']:.4f} | FPR: {best['fpr']:.4f}")

# uji threshold terpilih di test
if best_threshold is not None:
    test_score = -model.decision_function(X_test)
    y_best = (test_score >= best_threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_best, labels=[0, 1]).ravel()
    tpr = tp / (tp + fn) if (tp + fn) else 0.0
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    print(f"Test TPR: {tpr:.4f} | FPR: {fpr:.4f}")

Best threshold (val): 0.030574
Validation TPR: 1.0000 | FPR: 0.0037
Test TPR: 0.9821 | FPR: 0.0041


`Post-Filtering & Evaluasi Berbasis Event`

In [17]:
# Post-filtering: anggap anomali valid jika muncul >= N menit berturut-turut
min_consecutive = 3

# prediksi raw berdasarkan threshold terpilih (fallback ke threshold bawaan jika None)
if best_threshold is not None:
    test_score = -model.decision_function(X_test)
    y_test_raw = (test_score >= best_threshold).astype(int)
else:
    y_test_raw = (model.predict(X_test) == -1).astype(int)

test_eval = test_df.copy()
test_eval["pred_raw"] = y_test_raw
test_eval = test_eval.sort_values(["node_id", "recorded_at"])

test_eval["pred_filtered"] = (
    test_eval.groupby("node_id")["pred_raw"]
    .transform(
        lambda s: s.rolling(min_consecutive, min_periods=min_consecutive)
        .sum()
        .ge(min_consecutive)
        .astype(int)
    )
    .fillna(0)
    .astype(int)
 )

# evaluasi point-level setelah post-filtering
for col in ["pred_raw", "pred_filtered"]:
    cm = confusion_matrix(y_test, test_eval[col], labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    tpr = tp / (tp + fn) if (tp + fn) else 0.0
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    print(f"{col} -> TPR: {tpr:.4f} | FPR: {fpr:.4f}")

# ekstrak event prediksi dari label point-level
def extract_events(df: pd.DataFrame, pred_col: str) -> pd.DataFrame:
    events = []
    for node_id, grp in df.groupby("node_id", sort=False):
        grp = grp.sort_values("recorded_at")
        in_event = False
        start_time = None
        last_time = None
        for row in grp.itertuples():
            pred = getattr(row, pred_col)
            ts = row.recorded_at
            if pred == 1 and not in_event:
                in_event = True
                start_time = ts
            if pred == 0 and in_event:
                events.append({"node_id": node_id, "start_time": start_time, "end_time": last_time})
                in_event = False
                start_time = None
            last_time = ts
        if in_event and start_time is not None:
            events.append({"node_id": node_id, "start_time": start_time, "end_time": last_time})
    return pd.DataFrame(events)

# ambil event anomali yang berada di rentang waktu test
test_start = test_eval["recorded_at"].min()
test_end = test_eval["recorded_at"].max()
true_events = anomalies[(anomalies["end_time"] >= test_start) & (anomalies["start_time"] <= test_end)].copy()
pred_events = extract_events(test_eval, "pred_filtered")

# hitung overlap event untuk precision/recall berbasis event
def has_overlap(event_row, events_df: pd.DataFrame) -> bool:
    subset = events_df[events_df["node_id"] == event_row.node_id]
    if subset.empty:
        return False
    return bool((subset["start_time"] <= event_row.end_time).any() and (subset["end_time"] >= event_row.start_time).any())

if not true_events.empty:
    true_events["hit"] = true_events.apply(lambda r: has_overlap(r, pred_events), axis=1)
    event_recall = true_events["hit"].mean()
else:
    event_recall = 0.0

if not pred_events.empty:
    pred_events["hit"] = pred_events.apply(lambda r: has_overlap(r, true_events), axis=1)
    event_precision = pred_events["hit"].mean()
    false_event_alerts = int((~pred_events["hit"]).sum())
else:
    event_precision = 0.0
    false_event_alerts = 0

print(f"Event-level Precision: {event_precision:.4f}")
print(f"Event-level Recall: {event_recall:.4f}")
print(f"False event alerts: {false_event_alerts}")

pred_raw -> TPR: 0.9821 | FPR: 0.0041
pred_filtered -> TPR: 0.9107 | FPR: 0.0037
Event-level Precision: 1.0000
Event-level Recall: 1.0000
False event alerts: 0


`Validation Report`

In [18]:
# distribusi label
print("Jumlah kelas (label=0 normal, label=1 anomali):")
display(label_dist)

# TPR/FPR validation dengan threshold terpilih
val_score = -model.decision_function(X_val)
y_val_hat = (val_score >= best_threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_val, y_val_hat, labels=[0, 1]).ravel()
tpr = tp / (tp + fn) if (tp + fn) else 0.0
fpr = fp / (fp + tn) if (fp + tn) else 0.0
print(f"Validation TPR: {tpr:.4f} | FPR: {fpr:.4f}")

Jumlah kelas (label=0 normal, label=1 anomali):


,train,val,test
label,,,
0,35110,7512,7504
1,170,48,56


Validation TPR: 1.0000 | FPR: 0.0037


### **Save Model**

In [20]:
# simpan model dan metadata threshold untuk inferensi di API
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

artifact = {
    "model": model,
    "feature_cols": feature_cols,
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "threshold": best_threshold,
    "threshold_target": {"tpr": 0.95, "fpr": 0.05},
    "post_filtering": {"min_consecutive": min_consecutive},
    "contamination": contamination,
    "split_time": {
        "train": str(train_split),
        "val": str(val_split),
    },
}

joblib.dump(artifact, MODEL_DIR / "isolation_forest.joblib")
print("Saved:", MODEL_DIR / "isolation_forest.joblib")

Saved: models\isolation_forest.joblib
